# Detecting Faces

Once the webcam is running, Codetto can pass the video to a small **neural network** that looks for faces in every frame. This model is called the **Face Detector**. It tells you *where* each face is, drawn as a rectangle called a **bounding box**, and how confident it is.

Face detection is not the same as **face recognition**. Recognition tries to work out *who* a person is. Detection only says "there is a face here" - it has no idea whose face it is.

Like the camera itself, the model runs **on your own computer** inside the browser. No video and no face data is uploaded anywhere.

Run the cell below. Your browser will ask for webcam permission — click **Allow**. A box should appear around your face for about eight seconds.

In [ ]:
from codetto import cv, graphics
import time

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_face_detector(camera)

try:
  end = time.time() + 8
  while time.time() < end:
    faces = detector.get_detections()
    canvas.draw_bounding_boxes(faces)
finally:
  detector.stop()
  camera.stop()

# The Detector Object

Face detection is possible with these three lines of code:

- `cv.start_camera(canvas)` turns the webcam on, and displays the feed on the canvas in the notebook.
- `cv.start_face_detector(camera)` starts the face model and hands you back a **detector object**.
- `detector.get_detections()` asks the detector for the faces it can see *right now*, returning a list.

Each time round the loop, the `canvas.draw_bounding_boxes(...)` method is called to paint the boxes over the video.

# What a Detection Contains

Every face detection returned is a **dictionary** with five useful numbers:

| Field | Meaning |
|---|---|
| `x`, `y` | Top-left corner of the box, in pixels |
| `w`, `h` | Width and height of the box, in pixels |
| `confidence` | How sure the model is, from `0.0` to `1.0` |

The cell below runs the same loop but also prints those numbers to the console every 240 frames.

Try moving closer, tilting your head, or half-covering your face, and watch how the numbers change. Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_face_detector(camera)

frame = 0
try:
  while True:
    faces = detector.get_detections()
    canvas.draw_bounding_boxes(faces)
    if frame % 240 == 0:
      print(f'--- {len(faces)} face(s) ---')
      for i, face in enumerate(faces):
        print(f"  Face {i + 1}: x={face['x']} y={face['y']} "
            f"size={face['w']}x{face['h']} confidence={face['confidence']:.0%}")
    frame += 1
finally:
  detector.stop()
  camera.stop()

# Confidence Scores

The `confidence` value is the model's own guess at how likely it is that the box really contains a face. `0.95` means "very sure", `0.30` means "probably, but I might be wrong".

The **confidence threshold** is a cut-off you choose. Every detection above the threshold is kept; the others are discarded.

- A **high** threshold (like `0.90`) shows only the boxes the model is very sure about.
- A **low** threshold (like `0.20`) shows more boxes, including shaky ones on things that are not faces.

Try experimenting with the threshold. (You'll need to re-run the cell every time you adjust the threshold.) Can you make the detector lose your face?

In [ ]:
from codetto import cv, graphics

MIN_CONFIDENCE = 0.7 #@param {type:"slider", min:0.1, max:1.0, step:0.05}

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_face_detector(camera)

try:
  while True:
    faces = detector.get_detections()
    sure = [face for face in faces if face['confidence'] >= MIN_CONFIDENCE]
    canvas.draw_bounding_boxes(sure)
finally:
  detector.stop()
  camera.stop()

# Counting Faces

Because `get_detections()` returns a list, `len(faces)` tells you how many faces the model can see at this moment. That number goes up and down as people move in and out of the frame.

The cell below watches that count and prints a line to the console **only when it changes**, so the output stays readable. Hold a photo up next to your own face, or get a friend to lean in, and watch the count jump to two.

Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_face_detector(camera)

last_count = -1
try:
  while True:
    faces = detector.get_detections()
    canvas.draw_bounding_boxes(faces)
    count = len(faces)
    if count != last_count:
      print(f'Now seeing {count} face(s)')
      last_count = count
finally:
  detector.stop()
  camera.stop()

# Check Your Understanding